# Word LLM Translation Workflow — Primary Translation Retry Stage

- **Workflow stage:** `primary_retry_completed` (written at the end of this notebook if retries are performed)
- **Input checkpoint:** primary translation checkpoint from Notebook 1
- **Source language:** loaded from checkpoint metadata
- **Target language:** loaded from checkpoint metadata
- **Primary translator model:** loaded from checkpoint metadata
- **Primary system message:** loaded from checkpoint metadata
- **Purpose of this notebook:** rerun only those batches that still contain `primary_error`

## Purpose of this notebook
This notebook performs a targeted retry pass for batches that failed during the primary translation stage. It reuses the same source language, target language, model name, and primary system message that were recorded in the saved checkpoint so that retry behavior remains consistent with the original primary translation run.

## Metadata flow
This notebook:

- loads `metadata` and `elements` from the primary translation checkpoint
- reuses the saved primary translation settings from metadata
- retries only batches that still contain `primary_error`
- saves an updated checkpoint with both `metadata` and `elements`

## Notes
- This notebook is conditional. It is only needed if the primary translation checkpoint contains one or more `primary_error` values.
- If there are no `primary_error` values, you can skip this notebook.
- The retry pass mutates the original element records in place so successful reruns update the canonical workflow state.

## Setup
### List contents of */checkpoints* subdirectory

In [15]:
# show json files in the /checkpoints subdirectory
import importlib
import workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)

checkpoints_dir = workflow_helpers.get_checkpoints_dir()
json_files = workflow_helpers.list_json_files(checkpoints_dir)

Found JSON files:
- elements_batched_20260403_2247.json
- primary_translation_completed_20260403_2251.json


### Load saved json state file
Run these cells at the beginning and do not reload.  
Enter the json filename you wish to use in the variable below.

In [16]:
# Load checkpoint state (metadata + elements)

import os
from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

# Define the checkpoint file
checkpoint_dir = "checkpoints"
json_file = "primary_translation_completed_20260403_2251.json"

# Full path
checkpoint_path = os.path.join(checkpoint_dir, json_file)

# Load metadata and elements
metadata, elements = workflow_helpers.load_elements_checkpoint(checkpoint_path)

print("Loaded checkpoint:", checkpoint_path)
print("Stage:", metadata.get("stage"))
print("Source language:", metadata.get("source_language"))
print("Target language:", metadata.get("target_language"))
print("Primary model:", metadata.get("primary_model_name"))
print("Primary system message (truncated):", metadata.get("primary_system_message", "")[:100])
print(f"Loaded checkpoint with {len(elements)} elements.")
print("Example entry:")
elements[0]

Loaded checkpoint: checkpoints\primary_translation_completed_20260403_2251.json
Stage: primary_translation_completed
Source language: English
Target language: Traditional Chinese
Primary model: gemini-3.1-pro-preview
Primary system message (truncated): You are a professional translator of Biblical education materials for a Protestant/Evangelical audie
Loaded checkpoint with 29 elements.
Example entry:


{'element_number': 1,
 'element_type': 'paragraph',
 'word_style': 'h1',
 'text': 'Introduction',
 'element_id': '8633e724fc99',
 'tokens': 1,
 'batch_number': 1,
 'primary_translation': '簡介',
 'primary_translation_model': 'gemini-3.1-pro-preview',
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

### Start counter
Run this only one time

In [17]:
# do not re-run this cell
retry_counter = 1

## Set up languages and LLM models

In [18]:
# Pull primary translation settings forward from checkpoint metadata

source_language = metadata.get("source_language")
target_language = metadata.get("target_language")
gemini_model_name = metadata.get("primary_model_name")
primary_system_message = metadata.get("primary_system_message")

missing = [
    name for name, value in {
        "source_language": source_language,
        "target_language": target_language,
        "gemini_model_name": gemini_model_name,
        "primary_system_message": primary_system_message,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        f"Missing required metadata field(s) in checkpoint: {', '.join(missing)}"
    )

print("Source language:", source_language)
print("Target language:", target_language)
print("Primary model:", gemini_model_name)
print("Primary system message (truncated):", primary_system_message[:100])

Source language: English
Target language: Traditional Chinese
Primary model: gemini-3.1-pro-preview
Primary system message (truncated): You are a professional translator of Biblical education materials for a Protestant/Evangelical audie


In [19]:
# Reality check: inspect the saved prompt reused for primary-translation retries

print("Primary system message loaded from metadata.")
print("Length:", len(primary_system_message))
print("Contains source language:", source_language in primary_system_message)
print("Contains target language:", target_language in primary_system_message)
print("\nPreview:\n")
print(primary_system_message[:500])

Primary system message loaded from metadata.
Length: 3854
Contains source language: True
Contains target language: True

Preview:

You are a professional translator of Biblical education materials for a Protestant/Evangelical audience.

You will receive a single JSON object with the following structure:

{
  "elements": [
    {
      "id": "<string>",
      "text": "<chunked Markdown in English>"
    },
    ...
  ]
}

Each `text` field is a small chunk of Markdown in English. For each element, you must translate the natural-language prose into Traditional Chinese while preserving formatting and structure.

You must respond 


### Initialize Gemini model

In [20]:
# Load Gemini client/model interface
# This notebook only uses Gemini for the primary translation stage.

import importlib
import workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)

gemini_client = workflow_helpers.initialize_gemini_client()

print("Gemini client initialized.")
print("Primary model:", gemini_model_name)

Gemini client initialized successfully: <google.genai.client.Client object at 0x000001D7AC32B910>
Gemini client initialized.
Primary model: gemini-3.1-pro-preview


## Check for primary failures and re-run primary translation
**Important:** Be sure to run all cells below before retrying again so that the retry counter updates

In [21]:
print(f"retry attempt: {str(retry_counter)}")

retry attempt: 1


In [22]:
error_counts = sorted({
    el["batch_number"]
    for el in elements
    if el.get("primary_error")
}
                     )

print(f"number of batches with primary error: {len(error_counts)}") 
print(error_counts)

number of batches with primary error: 0
[]


In [23]:
from importlib import reload
import workflow_helpers
from datetime import datetime
import time

workflow_helpers = reload(workflow_helpers)

# 1. Identify all batches that still have a primary_error
primary_error_batches = sorted({
    el["batch_number"]
    for el in elements
    if el.get("primary_error")
})

print(f"Retrying {len(primary_error_batches)} batches:", primary_error_batches)

# 2. Retry loop: processes batches one at a time, mutating `elements` in-place
for i, batch_number in enumerate(primary_error_batches, start=1):

    # IMPORTANT: do NOT copy — we need references to the original dicts
    batch_elements = [
        el for el in elements
        if el.get("batch_number") == batch_number
    ]

    now = datetime.now().strftime("%H:%M:%S")
    print(
        f"[{now}] Retrying batch {batch_number} "
        f"({i}/{len(primary_error_batches)}) with {len(batch_elements)} elements..."
    )

    workflow_helpers.run_primary_translation(
        elements=batch_elements,              # in-place update
        gemini_client=gemini_client,
        primary_system_message=primary_system_message,
        primary_model_name=gemini_model_name,
        verbose=False,                        # we handle the printing manually
        print_status_every_n_batches=1
    )

    # optional pause to avoid hammering the API
    time.sleep(0.5)

now = datetime.now().strftime("%H:%M:%S")
print(f"[{now}] Retry loop complete.")

Retrying 0 batches: []
[22:53:12] Retry loop complete.


In [24]:
# display re-try failures
# tabulate re-try failures

remaining_error_batches = sorted(
    {el["batch_number"] for el in elements if el.get("primary_error")}
)

print(f"number of batches that failed on re-try: {len(remaining_error_batches)}") 
print(remaining_error_batches)


number of batches that failed on re-try: 0
[]


In [25]:
# save intermediate state to json

from importlib import reload
import workflow_helpers

# Reload the updated helper file
workflow_helpers = reload(workflow_helpers)

# Update metadata for this retry-stage checkpoint
metadata["stage"] = workflow_helpers.WORKFLOW_STAGES["primary_retry_completed"]

# Save the checkpoint with metadata + elements
checkpoint_path = workflow_helpers.save_elements_checkpoint(
    elements=elements,
    base_filename=f"{workflow_helpers.WORKFLOW_STAGES['primary_retry_completed']}_{retry_counter}",
    metadata=metadata,
)

retry_counter += 1

print("Checkpoint saved to:", checkpoint_path)
print("Metadata saved:")
print(metadata)

Checkpoint saved to: checkpoints\primary_retry_completed_1_20260403_2253.json
Metadata saved:
{'stage': 'primary_retry_completed', 'docxfilename': 'custom_word_styles_example.docx', 'source_language': 'English', 'target_language': 'Traditional Chinese', 'primary_model_name': 'gemini-3.1-pro-preview', 'evaluation_model_name': None, 'fallback_model_name': None, 'primary_system_message': 'You are a professional translator of Biblical education materials for a Protestant/Evangelical audience.\n\nYou will receive a single JSON object with the following structure:\n\n{\n  "elements": [\n    {\n      "id": "<string>",\n      "text": "<chunked Markdown in English>"\n    },\n    ...\n  ]\n}\n\nEach `text` field is a small chunk of Markdown in English. For each element, you must translate the natural-language prose into Traditional Chinese while preserving formatting and structure.\n\nYou must respond with a single valid JSON object of the form:\n\n{\n  "elements": [\n    {\n      "id": "<same i

In [26]:
import json

with open(checkpoint_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Top-level type:", type(data).__name__)
print("Keys:", list(data.keys()) if isinstance(data, dict) else None)
print("Metadata:", data.get("metadata") if isinstance(data, dict) else None)

Top-level type: dict
Keys: ['metadata', 'elements']
Metadata: {'stage': 'primary_retry_completed', 'docxfilename': 'custom_word_styles_example.docx', 'source_language': 'English', 'target_language': 'Traditional Chinese', 'primary_model_name': 'gemini-3.1-pro-preview', 'evaluation_model_name': None, 'fallback_model_name': None, 'primary_system_message': 'You are a professional translator of Biblical education materials for a Protestant/Evangelical audience.\n\nYou will receive a single JSON object with the following structure:\n\n{\n  "elements": [\n    {\n      "id": "<string>",\n      "text": "<chunked Markdown in English>"\n    },\n    ...\n  ]\n}\n\nEach `text` field is a small chunk of Markdown in English. For each element, you must translate the natural-language prose into Traditional Chinese while preserving formatting and structure.\n\nYou must respond with a single valid JSON object of the form:\n\n{\n  "elements": [\n    {\n      "id": "<same id value as input>",\n      "tran

#### Additional inspection cells

In [27]:
elements[0]

{'element_number': 1,
 'element_type': 'paragraph',
 'word_style': 'h1',
 'text': 'Introduction',
 'element_id': '8633e724fc99',
 'tokens': 1,
 'batch_number': 1,
 'primary_translation': '簡介',
 'primary_translation_model': 'gemini-3.1-pro-preview',
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

In [28]:
missing_primary_translation = [el for el in elements if el.get("primary_translation") is None]

print(f"Elements still missing primary translation: {len(missing_primary_translation)}")
for el in missing_primary_translation:
    print(f"  element_number={el['element_number']}, id={el['element_id']}")


Elements still missing primary translation: 0


## Notebook checkpoint and handoff

This notebook performed a targeted retry pass for failed primary-translation batches by completing the following steps:

- loaded the primary translation checkpoint from Notebook 1
- loaded workflow metadata and schema-complete element state from the checkpoint structure
- pulled `source_language`, `target_language`, `primary_model_name`, and `primary_system_message` forward from checkpoint metadata
- identified batches that still contained `primary_error`
- reran only those failed batches using the same primary translation settings as the original run
- updated the canonical element state in place for any successful retries
- saved the resulting intermediate state to a timestamped JSON checkpoint together with workflow metadata

### Output of this notebook
The main output is a timestamped JSON checkpoint containing:

- a top-level `metadata` block
- an `elements` list containing the updated schema state after retry attempts

This file is intended to be used as input for the next notebook.

### Metadata saved at this stage
The checkpoint metadata currently records:

- `stage = primary_retry_completed`
- `source_language`
- `target_language`
- `primary_model_name`
- `primary_system_message`
- placeholder fields for `evaluation_model_name`, `fallback_model_name`, `evaluation_system_message`, and `fallback_system_message`, which remain `None` at this stage

### Scope of this notebook
This notebook focuses only on repairing failed primary translations.

At this stage:

- only elements with `primary_error` are targeted for retry
- successful retries overwrite the prior primary-stage error state for those elements
- no evaluator logic is run here
- no fallback translation logic is run here
- `final` and `final_model` remain unassigned placeholders

### Next step
If all `primary_error` values have been cleared, proceed to the next analysis/evaluation stage.

Go to: `3_primary_translation_analysis.ipynb`